# Phase 2 FINAL - ExPSO vs Magnitude Pruning (50% Compression)

## Setup:
1. Upload checkpoints to `checkpoints/` folder
2. Run all cells

In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchattacks
import numpy as np
import json
import warnings
import time
warnings.filterwarnings('ignore')

from utils import (
    device, get_cifar10_loaders, evaluate, train_one_epoch,
    save_checkpoint, load_checkpoint, count_zero_params
)
from expsoptimizer import ExPSOPruner, magnitude_prune

print(f'Device: {device}')

Device: cuda


In [2]:
# Find checkpoint
checkpoint_paths = [
    'checkpoints/resnet18_standard_best.pth',
    'resnet18_standard_best.pth',
]

CHECKPOINT_PATH = None
for path in checkpoint_paths:
    if os.path.exists(path):
        CHECKPOINT_PATH = path
        print(f'Found checkpoint: {path}')
        break

if CHECKPOINT_PATH is None:
    raise FileNotFoundError('Checkpoint not found!')

# Load data
trainloader, testloader = get_cifar10_loaders()
print('Data loaded successfully')

Found checkpoint: checkpoints/resnet18_standard_best.pth
Data loaded successfully


In [3]:
# ==========================================================================
# FINAL CONFIGURATION
# ==========================================================================
USE_STANDARD_MODEL = True  # Start from standard (creates robustness)
TRAIN_EPOCHS = 80  # Longer fine-tuning
PRUNING_RATIO = 0.50  # 50% total model sparsity
EXPSO_ADVERSARIAL_FITNESS = True  # Use adversarial-aware fitness
LR_WARMUP_EPOCHS = 5  # Learning rate warmup
BASE_LR = 0.05  # Base learning rate for fine-tuning

eps = 8/255
alpha = 2/255
steps = 20

print(f"\n{'='*60}")
print("FINAL CONFIGURATION")
print(f"{'='*60}")
print(f"Base Model: {'Standard' if USE_STANDARD_MODEL else 'Adversarial'}")
print(f"Pruning Ratio: {PRUNING_RATIO:.0%}")
print(f"Target Total Sparsity: ~50%")
print(f"Fine-tune Epochs: {TRAIN_EPOCHS}")
print(f"Adversarial Fitness: {EXPSO_ADVERSARIAL_FITNESS}")
print(f"{'='*60}")


FINAL CONFIGURATION
Base Model: Standard
Pruning Ratio: 50%
Target Total Sparsity: ~50%
Fine-tune Epochs: 80
Adversarial Fitness: True


In [4]:
# Load base model
print('\nLoading base model...')
base_model = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
base_model = load_checkpoint(base_model, CHECKPOINT_PATH)
base_model = base_model.float()

clean_base = evaluate(base_model, testloader)
robust_base = evaluate(base_model, testloader, 
    atk=torchattacks.PGD(base_model, eps=eps, alpha=alpha, steps=steps))

print(f'Base Model Results:')
print(f'  Clean: {clean_base:.2f}%')
print(f'  Robust (PGD-20): {robust_base:.2f}%')
print(f'  Sparsity: {count_zero_params(base_model):.2f}%')


Loading base model...
Base Model Results:
  Clean: 88.77%
  Robust (PGD-20): 0.37%
  Sparsity: 0.00%


## 1. Magnitude Pruning

In [5]:
print('\n' + '='*60)
print('MAGNITUDE PRUNING (Conv + FC)')
print('='*60)

# Reload fresh model
model_mag = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
model_mag = load_checkpoint(model_mag, CHECKPOINT_PATH)
model_mag = model_mag.float()

# Apply magnitude pruning (handles both Conv and FC)
start_time = time.time()
print(f'\nPruning at {PRUNING_RATIO:.0%}...')
model_mag = magnitude_prune(model_mag, pruning_ratio=PRUNING_RATIO, device=device)
mag_prune_time = time.time() - start_time

# Evaluate before fine-tuning
clean_before = evaluate(model_mag, testloader)
robust_before = evaluate(model_mag, testloader, 
    atk=torchattacks.PGD(model_mag, eps=eps, alpha=alpha, steps=steps))
print(f'\nBefore fine-tuning:')
print(f'  Clean: {clean_before:.2f}%')
print(f'  Robust: {robust_before:.2f}%')


MAGNITUDE PRUNING (Conv + FC)

Pruning at 50%...
Magnitude pruning: Found 16 prunable layers
Achieved sparsity: 50.00%
Conv Sparsity: 50.00%
FC Sparsity: 0.00% (Safely Excluded)
Total Sparsity: 50.00%

Before fine-tuning:
  Clean: 28.41%
  Robust: 0.00%


In [6]:
# Fine-tune magnitude-pruned model
print('\nFine-tuning (80 epochs with LR warmup)...')

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_mag.parameters(), lr=BASE_LR, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS)
train_atk = torchattacks.PGD(model_mag, eps=eps, alpha=alpha, steps=10)

best_mag_robust = 0
best_mag_clean = 0
best_mag_epoch = 0

for epoch in range(1, TRAIN_EPOCHS + 1):
    # LR warmup
    if epoch <= LR_WARMUP_EPOCHS:
        lr_scale = epoch / LR_WARMUP_EPOCHS
        for pg in optimizer.param_groups:
            pg['lr'] = BASE_LR * lr_scale
    
    loss, acc = train_one_epoch(model_mag, trainloader, optimizer, criterion, epoch, atk=train_atk)
    
    if epoch > LR_WARMUP_EPOCHS:
        scheduler.step()
    
    if epoch % 10 == 0 or epoch == TRAIN_EPOCHS:
        clean = evaluate(model_mag, testloader)
        robust = evaluate(model_mag, testloader, 
            atk=torchattacks.PGD(model_mag, eps=eps, alpha=alpha, steps=steps))
        print(f'Epoch {epoch}: Clean={clean:.2f}%, Robust={robust:.2f}%')
        
        if robust > best_mag_robust:
            best_mag_robust = robust
            best_mag_clean = clean
            best_mag_epoch = epoch
            save_checkpoint(model_mag, 'checkpoints/mag_final_best.pth')

print(f'\nMagnitude Best: Clean={best_mag_clean:.2f}%, Robust={best_mag_robust:.2f}% (Epoch {best_mag_epoch})')
print(f'Pruning time: {mag_prune_time:.1f}s')


Fine-tuning (80 epochs with LR warmup)...


Epoch 10: 100%|██████████| 391/391 [01:11<00:00,  5.45it/s, loss=1.76, acc=33.8]


Epoch 10: Clean=57.62%, Robust=33.55%


Epoch 20: 100%|██████████| 391/391 [01:14<00:00,  5.28it/s, loss=1.7, acc=36]   


Epoch 20: Clean=57.15%, Robust=31.88%


...

Epoch 60: Clean=65.44%, Robust=37.84%


Epoch 70: 100%|██████████| 391/391 [01:14<00:00,  5.24it/s, loss=1.5, acc=42.7] 


Epoch 70: Clean=68.20%, Robust=38.84%


Epoch 80: 100%|██████████| 391/391 [01:14<00:00,  5.26it/s, loss=1.44, acc=44.3]


Epoch 80: Clean=68.74%, Robust=38.88%

Magnitude Best: Clean=68.74%, Robust=38.88% (Epoch 80)
Pruning time: 0.1s


## 2. ExPSO-Guided Pruning

In [7]:
EXPSO_PARTICLES = 30
EXPSO_ITER = 60

print(f"ExPSO Particles: {EXPSO_PARTICLES}")
print(f"ExPSO Iterations: {EXPSO_ITER}")

ExPSO Particles: 30
ExPSO Iterations: 60


In [8]:
expert_ratios = [0.30, 0.50, 0.70]
expso_final_results = [] # To store the final "Best" stats for your table

In [9]:
for ratio in expert_ratios:
    print('\n' + '='*60)
    print(f'ExPSO-GUIDED PRUNING: RATIO {ratio:.0%}')
    print('='*60)

    # Reload fresh model for each expert
    model_expso = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
    model_expso = load_checkpoint(model_expso, CHECKPOINT_PATH)
    model_expso = model_expso.float()

    # ExPSO optimization
    pruner = ExPSOPruner(
        model=model_expso,
        pruning_ratio=ratio, # Dynamic ratio
        n_particles=EXPSO_PARTICLES,
        max_iter=EXPSO_ITER,
        batch_size=32,
        device=device,
        use_adversarial_fitness=EXPSO_ADVERSARIAL_FITNESS,
        epsilon=eps,
        alpha=alpha,
        attack_steps=5
    )

    print(f'Particles: {EXPSO_PARTICLES}')
    print(f'Iterations: {EXPSO_ITER}')
    print(f'Adversarial Fitness: {EXPSO_ADVERSARIAL_FITNESS}')

    start_time = time.time()
    model_expso = pruner.prune(model_expso, trainloader)
    expso_opt_time = time.time() - start_time

    # Evaluate before fine-tuning
    clean_before = evaluate(model_expso, testloader)
    robust_before = evaluate(model_expso, testloader,
        atk=torchattacks.PGD(model_expso, eps=eps, alpha=alpha, steps=steps))
    print(f'\nBefore fine-tuning:')
    print(f'  Clean: {clean_before:.2f}%')
    print(f'  Robust: {robust_before:.2f}%')
    print(f'ExPSO optimization: {expso_opt_time/60:.1f} minutes')


ExPSO-GUIDED PRUNING: RATIO 30%
Particles: 30
Iterations: 60
Adversarial Fitness: True
Identifying prunable layers...
Found 16 prunable layers (Conv2d + Linear)
Initializing swarm with 30 particles...
Initial best fitness: 4.2106
Running ExPSO optimization for ratio 0.30...


  2%|▏         | 1/60 [00:00<00:22,  2.59it/s]

Iter 0: Best fitness = 4.2106


 18%|█▊        | 11/60 [00:04<00:19,  2.46it/s]

Iter 10: Best fitness = 4.2106


...

Iter 40: Best fitness = 7.3124


 85%|████████▌ | 51/60 [00:20<00:03,  2.39it/s]

Iter 50: Best fitness = 7.3124


100%|██████████| 60/60 [00:24<00:00,  2.44it/s]


Final best fitness: 7.3124
Applying optimal pruning masks...
Achieved sparsity: 70.17%

Before fine-tuning:
  Clean: 19.71%
  Robust: 1.63%
ExPSO optimization: 0.5 minutes


In [10]:
# =========================================================================
# PHASE 2/3: WRAPPED EXPSO EXPERT LOOP (3 RATIOS)
# =========================================================================
for ratio in expert_ratios:

    # Fine-tune ExPSO model
    print(f'\nFine-tuning {ratio:.0%} Expert (80 epochs with LR warmup)...')

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model_expso.parameters(), lr=BASE_LR, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS)
    train_atk = torchattacks.PGD(model_expso, eps=eps, alpha=alpha, steps=10)

    best_expso_robust = 0
    best_expso_clean = 0
    best_expso_epoch = 0

    for epoch in range(1, TRAIN_EPOCHS + 1):
        if epoch <= LR_WARMUP_EPOCHS:
            lr_scale = epoch / LR_WARMUP_EPOCHS
            for pg in optimizer.param_groups:
                pg['lr'] = BASE_LR * lr_scale
        
        train_one_epoch(model_expso, trainloader, optimizer, criterion, epoch, atk=train_atk)
        
        if epoch > LR_WARMUP_EPOCHS:
            scheduler.step()
        
        if epoch % 10 == 0 or epoch == TRAIN_EPOCHS:
            clean = evaluate(model_expso, testloader)
            robust = evaluate(model_expso, testloader,
                atk=torchattacks.PGD(model_expso, eps=eps, alpha=alpha, steps=steps))
            print(f'Epoch {epoch}: Clean={clean:.2f}%, Robust={robust:.2f}%')
            
            if robust > best_expso_robust:
                best_expso_robust = robust
                best_expso_clean = clean
                best_expso_epoch = epoch
                # Dynamic filename for each expert
                save_checkpoint(model_expso, f'checkpoints/expso_expert_{int(ratio*100)}.pth')

    print(f'\nExPSO {ratio:.0%} Best: Clean={best_expso_clean:.2f}%, Robust={best_expso_robust:.2f}% (Epoch {best_expso_epoch})')
    
    # Store results for the final comprehensive table
    expso_final_results.append({
        'Method': f'ExPSO-{int(ratio*100)}',
        'Clean': best_expso_clean,
        'PGD-20': best_expso_robust,
        # Re-run full eval for the final summary stats
        'Sparsity': count_zero_params(model_expso)
    })

print("\n--- ALL K=3 EXPERTS SUCCESSFULLY TRAINED AND SAVED ---")



Fine-tuning 30% Expert (80 epochs with LR warmup)...


Epoch 10: 100%|██████████| 391/391 [01:14<00:00,  5.21it/s, loss=1.82, acc=31.8]


Epoch 10: Clean=51.92%, Robust=30.35%


Epoch 20: 100%|██████████| 391/391 [01:16<00:00,  5.14it/s, loss=1.77, acc=33.6]


Epoch 20: Clean=57.29%, Robust=33.33%


...

Epoch 60: Clean=66.02%, Robust=35.92%


Epoch 70: 100%|██████████| 391/391 [01:12<00:00,  5.42it/s, loss=1.52, acc=41.6]


Epoch 70: Clean=66.27%, Robust=37.20%


Epoch 80: 100%|██████████| 391/391 [01:11<00:00,  5.47it/s, loss=1.46, acc=43.1]


Epoch 80: Clean=68.15%, Robust=37.36%

ExPSO 70% Best: Clean=68.15%, Robust=37.36% (Epoch 80)

--- ALL K=3 EXPERTS SUCCESSFULLY TRAINED AND SAVED ---


## 3. Final Results

In [ ]:
print('\n' + '='*70)
print('PHASE 2 FINAL RESULTS (Consolidated Experts)')
print('='*70)

# 1. Define Evaluation Suite
def evaluate_all(model, name):
    print(f"Evaluating {name}...")
    clean = evaluate(model, testloader)
    pgd20 = evaluate(model, testloader, atk=torchattacks.PGD(model, eps=eps, alpha=alpha, steps=20))
    pgd100 = evaluate(model, testloader, atk=torchattacks.PGD(model, eps=eps, alpha=alpha, steps=100))
    fgsm = evaluate(model, testloader, atk=torchattacks.FGSM(model, eps=eps))
    sparsity = count_zero_params(model)
    return {'Method': name, 'Clean': clean, 'PGD-20': pgd20, 'PGD-100': pgd100, 
            'FGSM': fgsm, 'Sparsity': sparsity}

# 2. Setup Results List
results = []

# Base Model (Standard)
results.append({'Method': 'Base', 'Clean': clean_base, 'PGD-20': robust_base, 
                'PGD-100': robust_base, 'FGSM': robust_base, 'Sparsity': 0})

# 3. Load and Evaluate Magnitude Baseline
model_mag = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
model_mag = load_checkpoint(model_mag, 'checkpoints/mag_final_best.pth')
results.append(evaluate_all(model_mag, 'Magnitude'))


# 5. Print Comparison Table
print(f"\n{'Method':<12} {'Clean':>8} {'PGD-20':>8} {'PGD-100':>8} {'FGSM':>8} {'Sparse':>8}")
print('-'*70)
for r in results:
    print(f"{r['Method']:<12} {r['Clean']:>7.2f}% {r['PGD-20']:>7.2f}% {r['PGD-100']:>7.2f}% {r['FGSM']:>7.2f}% {r['Sparsity']:>7.1f}%")


PHASE 2 FINAL RESULTS (Consolidated Experts)
Evaluating Magnitude...
Evaluating ExPSO-30...
Evaluating ExPSO-50...
Evaluating ExPSO-70...

Method          Clean   PGD-20  PGD-100     FGSM   Sparse
----------------------------------------------------------------------
Base           88.77%    0.37%    0.37%    0.37%     0.0%
Magnitude      68.74%   38.85%   38.65%   43.72%    49.2%
ExPSO-30       65.96%   37.46%   37.21%   41.93%    69.0%
ExPSO-50       67.46%   37.40%   37.17%   42.28%    69.0%
ExPSO-70       68.15%   37.35%   37.01%   42.45%    69.0%

--- ExPSO-50 vs Magnitude (50%) ---
Clean: -1.28%
Robust: -1.45%
Sparsity: +19.83%

--- ALL 3 EXPERTS READY FOR PHASE 3 ---
Ready for Phase 3 (SME) and Phase 4 (MVC)


In [12]:
import torch
print("--- RAW DISK CHECK ---")
for r in [30, 50, 70]:
    path = f'checkpoints/expso_expert_{r}.pth'
    try:
        ckpt = torch.load(path, map_location='cpu')
        # Check accuracies stored in your results list vs the physical files
        zeros = 0
        total = 0
        for k, v in ckpt.items():
            if 'weight' in k:
                total += v.numel()
                zeros += (v == 0).sum().item()
        print(f"File {path}: Physical Sparsity = {100.0 * zeros / total:.2f}%")
    except Exception as e:
        print(f"Could not check {path}: {e}")


--- RAW DISK CHECK ---
File checkpoints/expso_expert_30.pth: Physical Sparsity = 34.78%
File checkpoints/expso_expert_50.pth: Physical Sparsity = 34.78%
File checkpoints/expso_expert_70.pth: Physical Sparsity = 34.78%
